<a href="https://colab.research.google.com/github/Telebotfaroff/Codespace/blob/main/File_Uploader_for_Google_Drive_V_1_217B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yt-dlp

In [ ]:
from tqdm import tqdm
import requests
from google.colab import drive
import os
from urllib.parse import urlparse, unquote
import os.path
import sys
import time

drive.mount('/content/drive')

# Define the destination folder in Google Drive
destination_folder = '/content/drive/MyDrive/Downloads/'

# Create the destination folder if it doesn't exist
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

try:
    while True:
        try:
            user_input = input("\n\nEnter the download link (or 'exit' to quit): ")
        except KeyboardInterrupt:
            print("\nKeyboard interrupt detected. Exiting the program.")
            break

        if user_input.lower() == 'exit':
            break

        try:
            response = requests.get(user_input, stream=True)
            response.raise_for_status()
        except requests.exceptions.MissingSchema:
            print("You have entered a wrong command. Please enter again.\n")
            continue

        file_size = int(response.headers.get('content-length', 0))

        file_size_mb = file_size / (1024 ** 2)
        file_size_gb = file_size / (1024 ** 3)

        file_size_str = f"{file_size_mb:.2f} MB / {file_size_gb:.2f} GB"

        progress_bar = tqdm(total=file_size, unit='B', unit_scale=True, leave=True)

        parsed_url = urlparse(user_input)
        file_name = unquote(os.path.basename(parsed_url.path))

        file_path = os.path.join(destination_folder, file_name)

        start_time = time.time()

        with open(file_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)
                    progress_bar.update(len(chunk))
                    sys.stdout.flush()

        progress_bar.close()

        end_time = time.time()
        total_time = end_time - start_time

        print()
        print(f"File name: {file_name}")
        print(f"File size: {file_size_str}")
        print(f"Total Time: {total_time:.2f} seconds")
        print(f"Saved location: {file_path}")
        print(f"File '{file_name}' uploaded successfully to '{destination_folder}'.\n")

except Exception as e:
    print(f"An error occurred: {str(e)}")
    print("Exiting the program.")

print("Exiting the program.")


: 

In [ ]:
import yt_dlp
from tqdm import tqdm
import requests
from google.colab import drive
import os
import sys
import time

drive.mount('/content/drive')

# Define the destination folder in Google Drive
destination_folder = '/content/drive/MyDrive/Downloads/'

if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

def get_video_info(url):
    ydl_opts = {'quiet': True, 'noplaylist': True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        return info.get('title', 'video'), info.get('ext', 'mp4')

try:
    while True:
        user_input = input("\nEnter the download link (or 'exit' to quit): ")
        if user_input.lower() == 'exit':
            break

        try:
            print("Fetching video information...")
            title, ext = get_video_info(user_input)
            # Sanitize filename
            clean_title = "".join([c for c in title if c.isalnum() or c in (' ', '.', '_')]).strip()
            file_name = f"{clean_title}.{ext}"

            response = requests.get(user_input, stream=True)
            response.raise_for_status()

            file_size = int(response.headers.get('content-length', 0))
            progress_bar = tqdm(total=file_size, unit='B', unit_scale=True, leave=True)
            file_path = os.path.join(destination_folder, file_name)

            start_time = time.time()
            with open(file_path, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        file.write(chunk)
                        progress_bar.update(len(chunk))

            progress_bar.close()
            total_time = time.time() - start_time

            print(f"\nDownloaded: {file_name}")
            print(f"Total Time: {total_time:.2f} seconds")
            print(f"Saved to: {file_path}")

        except Exception as e:
            print(f"An error occurred with this link: {e}")

except KeyboardInterrupt:
    print("\nStopped by user.")